# LAB 09 · Tokenizer và Label Alignment

Mục 8–11 và Experiment 1 ở mục 17.


Nếu thiếu thư viện, chạy trong một cell:

```python
%pip install "transformers==4.49.0" "torch>=2.6,<3" scikit-learn pandas matplotlib
```

Lần đầu chạy, chương trình tải ba split CoNLL-2003. Nếu đã có dữ liệu, đặt biến
môi trường `LAB09_DATA_DIR` trỏ tới thư mục chứa eng.train, eng.testa và eng.testb.

## 8. Subword Tokenization

WordPiece có thể đổi một từ gốc thành nhiều ID.
Ví dụ `play + ##ing` minh họa nguyên tắc trong đề.
Cell bên dưới in kết quả **playing** thực tế từ tokenizer đã chọn.

In [1]:
import os
from pathlib import Path
from collections import Counter
import random

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoTokenizer

from baseline_ner import LABEL2ID, LABELS, SEED, load_corpus

MODEL_ID = "google/bert_uncased_L-2_H-128_A-2"
MODEL_SOURCE = os.getenv("LAB09_MODEL_SOURCE", MODEL_ID)
if "LAB09_MODEL_SOURCE" not in os.environ and Path("checkpoint/tokenizer.json").is_file():
    MODEL_SOURCE = "checkpoint"
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True)
assert tokenizer.is_fast
IGNORE = -100

## 9. Label Alignment

**Words → nhãn theo từ → WordPiece → word_ids → nhãn theo token.**

CoNLL gán BIO cho từng **từ gốc**. `word_ids()` truy vết từng ID về chỉ số từ.
`None` xuất hiện ở token đặc biệt hoặc padding. Bảng giữ cả **word_label** và
**token_label** để thấy nhãn nào thực sự đi vào mô hình.

## 10. Alignment Strategies

| Strategy | Mảnh đầu của từ mang B-X | Các mảnh tiếp theo | Đơn vị giám sát |
| --- | --- | --- | --- |
| A · Chọn | B-X | -100 | Một mảnh mỗi từ |
| B · Lan truyền | B-X | I-X | Mọi mảnh |
| C · Theo pipeline | Theo pipeline | Theo pipeline | Theo pipeline |

**Chọn A cho train, validation, test và IE.** Mảnh đầu giữ nguyên BIO, gồm I-X và O.
Các mảnh tiếp theo, special tokens và pad có nhãn -100. A giữ mức giám sát như nhau
giữa từ một mảnh và từ nhiều mảnh và đưa dự đoán về đúng chuỗi theo từ để chấm span.
B tăng vị trí giám sát nhưng từ bị tách nhiều sẽ đóng góp nhiều lần vào loss.
Các kết quả thực nghiệm dùng A xuyên suốt.

In [2]:
def align_first_pieces(word_ids, word_tags):
    """Mục 9–11 — Strategy A, cùng cách gán cho cả ba split."""
    targets, kept = [], []
    previous = None
    for index, word_id in enumerate(word_ids):
        if word_id is None or word_id == previous:
            targets.append(IGNORE)
        else:
            targets.append(LABEL2ID[word_tags[word_id]])
            kept.append(index)
        previous = word_id
    return targets, kept

In [3]:
words = ["Tsvetana", "Pironkova", "visited", "Ouagadougou", "."]
tags = ["B-PER", "I-PER", "O", "B-LOC", "O"]
batch = tokenizer(
    [words, ["Washington", "."]],
    is_split_into_words=True,
    padding=True,
    truncation=True,
    max_length=256,
)
batch_word_tags = [tags, ["B-LOC", "O"]]

def show_alignment(batch_index):
    ids = batch["input_ids"][batch_index]
    word_ids = batch.word_ids(batch_index)
    token_tags, first_pieces = align_first_pieces(word_ids, batch_word_tags[batch_index])
    source_words = (words, ["Washington", "."])[batch_index]
    return pd.DataFrame({
        "token": tokenizer.convert_ids_to_tokens(ids),
        "input_id": ids,
        "attention_mask": batch["attention_mask"][batch_index],
        "word_id": pd.array(word_ids, dtype="Int64"),
        "word": [source_words[i] if i is not None else "—" for i in word_ids],
        "word_label": [batch_word_tags[batch_index][i] if i is not None else "—" for i in word_ids],
        "token_label_id": token_tags,
        "token_label": ["IGNORE" if i == IGNORE else LABELS[i] for i in token_tags],
    }), first_pieces

print(list(zip(words, tags)))
print("Tokenizer thực tế xử lý playing:", tokenizer.tokenize("playing"))
first_table, first_pieces = show_alignment(0)
pad_table, _ = show_alignment(1)
display(first_table, pad_table)
assert first_table.loc[first_pieces, "token_label"].tolist() == tags
assert pad_table.loc[pad_table["attention_mask"].eq(0), "token_label_id"].eq(IGNORE).all()

[('Tsvetana', 'B-PER'), ('Pironkova', 'I-PER'), ('visited', 'O'), ('Ouagadougou', 'B-LOC'), ('.', 'O')]
Tokenizer thực tế xử lý playing: ['playing']


,token,input_id,attention_mask,word_id,word,word_label,token_label_id,token_label
0,[CLS],101,1,<NA>,—,—,-100,IGNORE
1,ts,24529,1,0,Tsvetana,B-PER,1,B-PER
2,##vet,19510,1,0,Tsvetana,B-PER,-100,IGNORE
3,##ana,5162,1,0,Tsvetana,B-PER,-100,IGNORE
4,pi,14255,1,1,Pironkova,I-PER,2,I-PER
5,##ron,4948,1,1,Pironkova,I-PER,-100,IGNORE
6,##kova,18810,1,1,Pironkova,I-PER,-100,IGNORE
7,visited,4716,1,2,visited,O,0,O
8,ou,15068,1,3,Ouagadougou,B-LOC,5,B-LOC
9,##aga,16098,1,3,Ouagadougou,B-LOC,-100,IGNORE


,token,input_id,attention_mask,word_id,word,word_label,token_label_id,token_label
0,[CLS],101,1,<NA>,—,—,-100,IGNORE
1,washington,2899,1,0,Washington,B-LOC,5,B-LOC
2,.,1012,1,1,.,O,0,O
3,[SEP],102,1,<NA>,—,—,-100,IGNORE
4,[PAD],0,0,<NA>,—,—,-100,IGNORE
5,[PAD],0,0,<NA>,—,—,-100,IGNORE
6,[PAD],0,0,<NA>,—,—,-100,IGNORE
7,[PAD],0,0,<NA>,—,—,-100,IGNORE
8,[PAD],0,0,<NA>,—,—,-100,IGNORE
9,[PAD],0,0,<NA>,—,—,-100,IGNORE


## 11. Ignore Index

Gọi $S=\{i:y_i\ne -100\}$. Công thức đề dùng tổng
$-\sum_{i\in S}\log p(y_i\mid x)$. PyTorch lấy trung bình theo số vị trí hợp lệ,
$L=-\frac{1}{|S|}\sum_{i\in S}\log p(y_i\mid x)$.

Mảnh bị ignore vẫn có attention_mask=1 và tham gia self-attention. Nó có thể
ảnh hưởng đến hidden states được giám sát dù gradient trực tiếp theo logit tại
vị trí ignore bằng 0. Padding có thêm attention_mask=0.
Cell đối chiếu loss tính tay và PyTorch rồi kiểm tra gradient tại từng vị trí.

In [4]:
torch.manual_seed(SEED)
logits = torch.randn(len(first_table), len(LABELS), requires_grad=True)
target = torch.tensor(first_table["token_label_id"].tolist())
included = target.ne(IGNORE)
log_probabilities = logits.log_softmax(dim=-1)
manual = -log_probabilities[included, target[included]].mean()
loss = F.cross_entropy(logits, target, ignore_index=IGNORE)
torch.testing.assert_close(loss, manual)
loss.backward()
assert torch.count_nonzero(logits.grad[~included]).item() == 0
assert torch.count_nonzero(logits.grad[included]).item() > 0
display(pd.DataFrame({
    "token": first_table["token"], "attention": first_table["attention_mask"],
    "supervised": included.tolist(),
    "grad_L1_at_logits": logits.grad.abs().sum(dim=-1).tolist(),
}))
print(f"Loss PyTorch = {loss.item():.6f} · loss tính tay = {manual.item():.6f}")

Loss PyTorch = 2.455137 · loss tính tay = 2.455137


,token,attention,supervised,grad_L1_at_logits
0,[CLS],1,False,0.000000
1,ts,1,True,0.382905
2,##vet,1,False,0.000000
3,##ana,1,False,0.000000
4,pi,1,True,0.332188
5,##ron,1,False,0.000000
6,##kova,1,False,0.000000
7,visited,1,True,0.342264
8,ou,1,True,0.376648
9,##aga,1,False,0.000000


## 17. Experiment 1 — Tokenization Analysis

Lấy mẫu bằng seed 42 trên **train** gồm 20 câu ngắn (≤10 từ) và 20 câu dài (≥25 từ).
**Words** là số từ gốc. **Subwords** bao gồm từ chỉ có một mảnh và bỏ token đặc biệt.
**Từ bị tách** đếm số từ có hơn một mảnh. **Pieces tăng thêm** bằng số mảnh trừ số từ.
Với một nhóm, $R=\sum \text{subwords}/\sum \text{words}$.
Từ entity gồm cả B và I. Bảng trình bày số liệu theo câu, độ dài câu,
entity/non-entity, type và dạng từ.

In [5]:
corpus = load_corpus()

def tokenization_experiment(source, tokenizer, n_each=20):
    rng = random.Random(SEED)
    pools = {
        "Ngắn (≤10 từ)": [i for i, row in enumerate(source) if len(row["words"]) <= 10],
        "Dài (≥25 từ)": [i for i, row in enumerate(source) if len(row["words"]) >= 25],
    }
    sentences, word_rows = [], []
    for group, pool in pools.items():
        for row_id in sorted(rng.sample(pool, n_each)):
            row = source[row_id]
            encoding = tokenizer(row["words"], is_split_into_words=True, add_special_tokens=False)
            counts = Counter(encoding.word_ids())
            assert set(counts) == set(range(len(row["words"])))
            num_words = len(row["words"])
            num_pieces = len(encoding["input_ids"])
            num_split = sum(size > 1 for size in counts.values())
            sentences.append({
                "train_id": row_id + 1, "nhóm": group, "words": num_words,
                "subwords": num_pieces, "từ bị tách": num_split,
                "pieces tăng thêm": num_pieces - num_words, "R": num_pieces / num_words,
            })
            pieces_by_word = [[] for _ in row["words"]]
            for token, word_id in zip(encoding.tokens(), encoding.word_ids()):
                pieces_by_word[word_id].append(token)
            for i, (word, tag) in enumerate(zip(row["words"], row["tags"])):
                word_rows.append({
                    "train_id": row_id + 1, "word": word, "tag": tag,
                    "nhóm": "Non-entity" if tag == "O" else "Entity",
                    "entity_type": tag[2:] if tag != "O" else "O",
                    "subwords": counts[i], "bị tách": counts[i] > 1,
                    "pieces": " ".join(pieces_by_word[i]),
                    "dạng": ("có chữ số" if any(c.isdigit() for c in word)
                             else "có chữ viết hoa" if any(c.isupper() for c in word)
                             else "chữ thường" if word.isalpha() else "dấu / ký tự khác"),
                })
    return pd.DataFrame(sentences), pd.DataFrame(word_rows)


def summarize_words(frame, group):
    result = frame.groupby(group, sort=False).agg(
        words=("word", "size"), subwords=("subwords", "sum"), split_words=("bị tách", "sum")
    )
    result["R"] = result["subwords"] / result["words"]
    result["split_rate"] = result["split_words"] / result["words"]
    return result


sentences, per_word = tokenization_experiment(corpus["train"], tokenizer)
by_sentence = sentences.groupby("nhóm", sort=False).agg(
    sentences=("train_id", "size"), words=("words", "sum"),
    subwords=("subwords", "sum"), split_words=("từ bị tách", "sum"),
)
by_sentence["R"] = by_sentence["subwords"] / by_sentence["words"]
by_sentence["split_rate"] = by_sentence["split_words"] / by_sentence["words"]
length_groups = summarize_words(per_word, "nhóm")
word_types = summarize_words(per_word, "entity_type")
shapes = summarize_words(per_word, "dạng")
display(sentences.round(4), by_sentence.round(4), length_groups.round(4),
        word_types.round(4), shapes.round(4))
examples = (per_word[per_word["bị tách"]].sort_values("subwords", ascending=False)
            .drop_duplicates("word").groupby("nhóm", sort=False).head(6))
display(examples[["train_id", "word", "tag", "pieces", "subwords"]])

,train_id,nhóm,words,subwords,từ bị tách,pieces tăng thêm,R
0,422,Ngắn (≤10 từ),2,3,1,1,1.5000
1,490,Ngắn (≤10 từ),1,3,1,2,3.0000
2,520,Ngắn (≤10 từ),6,11,3,5,1.8333
3,1493,Ngắn (≤10 từ),8,12,2,4,1.5000
4,1640,Ngắn (≤10 từ),9,9,0,0,1.0000
5,1785,Ngắn (≤10 từ),4,4,0,0,1.0000
6,1916,Ngắn (≤10 từ),8,8,0,0,1.0000
7,2317,Ngắn (≤10 từ),3,4,1,1,1.3333
8,3563,Ngắn (≤10 từ),5,9,2,4,1.8000
9,3788,Ngắn (≤10 từ),2,6,1,4,3.0000


,sentences,words,subwords,split_words,R,split_rate
nhóm,,,,,,
Ngắn (≤10 từ),20,116,190,34,1.6379,0.2931
Dài (≥25 từ),20,605,709,64,1.1719,0.1058


,words,subwords,split_words,R,split_rate
nhóm,,,,,
Non-entity,604,705,52,1.1672,0.0861
Entity,117,194,46,1.6581,0.3932


,words,subwords,split_words,R,split_rate
entity_type,,,,,
O,604,705,52,1.1672,0.0861
ORG,41,63,12,1.5366,0.2927
MISC,16,17,1,1.0625,0.0625
PER,46,94,30,2.0435,0.6522
LOC,14,20,3,1.4286,0.2143


,words,subwords,split_words,R,split_rate
dạng,,,,,
có chữ viết hoa,153,235,49,1.5359,0.3203
chữ thường,417,434,14,1.0408,0.0336
có chữ số,48,107,22,2.2292,0.4583
dấu / ký tự khác,103,123,13,1.1942,0.1262


,train_id,word,tag,pieces,subwords
99,10852,10-0-59-1,O,10 - 0 - 59 - 1,7
104,10852,10-1-39-1,O,10 - 1 - 39 - 1,7
101,10852,10-0-42-0,O,10 - 0 - 42 - 0,7
115,13860,04/06/96,O,04 / 06 / 96,5
77,9319,1996-08-27,O,1996 - 08 - 27,5
47,3788,1996-08-24,O,1996 - 08 - 24,5
365,9086,Eletropaulo,B-ORG,el ##et ##rop ##au ##lo,5
155,610,Ferencvaros,B-ORG,fe ##ren ##c ##var ##os,5
42,3563,Mujtaba,I-PER,mu ##j ##ta ##ba,4
357,6190,Odnosum,I-PER,o ##d ##nos ##um,4


### 17.1. Những loại từ nào thường bị tách?

Trong 40 câu đã chọn, **39,3% từ entity** bị tách, so với **8,6% non-entity**.
Dạng chứa chữ số bị tách nhiều nhất (45,8%), tiếp theo là dạng có chữ hoa (32,0%).
Eletropaulo và Ferencvaros có 5 mảnh mỗi từ; ngày 04/06/96 cũng có 5 mảnh dù nhãn O.
Từ chỉ có chữ thường có tỷ lệ tách 3,4%. Chữ hoa ở đây mô tả từ CoNLL gốc,
vì WordPiece uncased chuẩn hóa trước khi tra vocab.

R của nhóm ngắn là 1,6379 và nhóm dài là 1,1719. Các kiểu từ như ngày
tháng, số liệu có dấu và tên riêng có thể kéo R lên trong một câu ngắn.
Kết quả so sánh mô tả hai nhóm được lấy mẫu theo seed 42.

In [6]:
for group, row in length_groups.iterrows():
    print(f"{group}: {int(row['split_words'])}/{int(row['words'])} từ bị tách "
          f"({row['split_rate']:.1%}), R = {row['R']:.4f}.")
for group, row in by_sentence.iterrows():
    print(f"{group}: R = {row['R']:.4f}; tỷ lệ tách = {row['split_rate']:.1%}.")
for form, row in shapes.sort_values("split_rate", ascending=False).iterrows():
    print(f"{form}: {int(row['split_words'])}/{int(row['words'])} từ bị tách "
          f"({row['split_rate']:.1%}).")

Non-entity: 52/604 từ bị tách (8.6%), R = 1.1672.
Entity: 46/117 từ bị tách (39.3%), R = 1.6581.
Ngắn (≤10 từ): R = 1.6379; tỷ lệ tách = 29.3%.
Dài (≥25 từ): R = 1.1719; tỷ lệ tách = 10.6%.
có chữ số: 22/48 từ bị tách (45.8%).
có chữ viết hoa: 49/153 từ bị tách (32.0%).
dấu / ký tự khác: 13/103 từ bị tách (12.6%).
chữ thường: 14/417 từ bị tách (3.4%).
